### 模型二：

In [3]:
import pandas as pd
from pulp import LpMinimize, LpProblem, LpVariable, lpSum, LpBinary,LpStatus
import collections
from itertools import product



file_path = "realdata_v5_2_new.xlsx"  
activities_df = pd.read_excel(file_path, sheet_name="ActivitiesInfo")
classroom_df = pd.read_excel(file_path, sheet_name="ClassroomsInfo")
student_courses_df = pd.read_excel(file_path, sheet_name="StudentsInfo")



In [4]:
df = pd.read_csv("processed_schedule_0.1_notRemote.csv")
# df = df[(df["Week"] == 11) & (df["Time_slot"] == 3)]

df

,Week,Time_slot,subject_candidates
0,1,1,"{11, 13}"
1,1,2,"{16, 18, 40, 5}"
2,1,3,{35}
3,1,4,{53}
4,1,5,{35}
...,...,...,...
89,12,1,{2}
90,12,2,{33}
91,12,3,{27}
92,12,7,{30}


In [5]:
courses = student_courses_df["Course_ID"]
courses = set([int(num) for row in courses for num in row.split(', ')])

classrooms = classroom_df["Classroom_ID"]
classrooms = set(int(row) for row in classrooms)

activity_types = activities_df["Activity_Type"]
activity_types = set(activity_types)

In [6]:
# Many parameters in Model 2 are derived by reducing the dimensions of parameters in Model 1. 
# For example, Model 1 has a parameter Ysai, while Model 2 has a parameter Ysi, which removes the dimension 'a'.

# Parameters
Pc = [row["Classroom_ID"] for _, row in classroom_df.iterrows() if row["Has_Computers"] == 1]
Ysi = {(row["Course_ID"], row["Activity_Type"]): 1 for _, row in activities_df.iterrows()}
Gs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Separation"] == 1}
Cs = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Computers"] == 1}
Ts = {row["Course_ID"]: 1 for _, row in activities_df.iterrows() if row["Requires_Tables"] == 1}
Tc = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Has_Tables"] == 1}
Act = {(row["Classroom_ID"], int(w), int(t)): 1 for _, row in classroom_df.iterrows() for w in row["Available_Weeks"].split(', ') for t in row["Time_Slot"].split(', ')}
Ic = {row["Classroom_ID"]: 1 for _, row in classroom_df.iterrows() if row["Is_Isolated"] == 1}
Ns = {row["Course_ID"]: row["Num_Students"] for _, row in activities_df.iterrows()}
Msg = collections.defaultdict(int)
for _, row in student_courses_df.iterrows():
    class_id = row["Class_ID"]
    course_ids = list(map(int, str(row["Course_ID"]).split(', ')))  # Convert to a list of integers
    for course_id in course_ids:
        Msg[(course_id, class_id)] += 1  # Directly use course_id from the row

class_courses = collections.defaultdict(set)
for (course, class_id), _ in Msg.items():
    class_courses[course].add(class_id)

Capacity = {row["Classroom_ID"]: row["Capacity"] for _, row in classroom_df.iterrows()}
Occupancy_rate = {1: 1, 2: 0.9, 3: 0}
CAPci = {(c, i): int(Capacity[c] * Occupancy_rate[i]) for c, i in product(classrooms, activity_types)}

In [7]:
import csv
with open("results-两个变化叠加.csv", "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Week", "Time_slot", "subject_candidates", "result","value"])

results = []

In [8]:
for count in range(len(df)):
    week = df.iloc[count]["Week"]
    time_slot = df.iloc[count]["Time_slot"]
    subjects = df["subject_candidates"].tolist()
    subjects = [int(i.strip('{}')) for i in subjects[count].split(', ')]
    # 2. Define the optimization model
    model = LpProblem(name="classroom_assignment", sense=LpMinimize)

    z = {(c, s): LpVariable(f"z_{c}_{s}", cat=LpBinary) for c in classrooms for s in subjects}
    w = {(c, s, g): LpVariable(f"w_{c}_{s}_{g}", cat=LpBinary) for c in classrooms for s in subjects for g in class_courses[s]}
   
    # Add slack variables for capacity relaxation
    eta = {(s): LpVariable(f"eta_{s}", lowBound=0) for s in subjects}
    lamda = 20
    beta = 10
    # Add virtual classroom variables
    z_v = {(s): LpVariable(f"z_v_{s}", cat=LpBinary) for s in subjects}

    # Objective function
    model += (
        lpSum(z[c, s] for c in classrooms for s in subjects) + 
        lamda * lpSum(eta[s] for s in subjects) +
        beta * lpSum(z_v[s] for s in subjects),
        "Minimize_Classroom_Usage"
    )

    # A.13 - Ensure classroom capacity is sufficient
    for s in subjects:
        model += lpSum(z[c, s] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) + eta[s] >= Ns[s] * (1 - z_v[s]), f"Capacity_Constraint_{s}"
        model += eta[s] <= 0.15 * Ns[s]

    # A.14 - Ensure classroom capacity is sufficient for each class
    for s in subjects:
        if Ts.get(s, 0) == 1:
            for g in class_courses[s]:
                model += lpSum(w[c, s, g] * sum(CAPci.get((c, i), 0) * Ysi.get((s, i), 0) for i in activity_types) for c in classrooms) >= Msg.get((s, g), 0) * (1 - z_v[s]), f"Classroom_Capacity_{s}_{g}"

    # A.15 - Prevent classrooms from being reused
    for c in classrooms:
        model += lpSum(z[c, s] for s in subjects) <= 1, f"Single_Assignment_{c}"

    # Add virtual classroom constraints
    for s in subjects:
        for c in classrooms:
            model += z_v[s] + z[c, s] <= 1, f"Virtual_Assignment_{s}_{c}"
    
    # A.16 - Only allow available classrooms to be assigned
    for c in classrooms:
        for s in subjects:
            model += z[c, s] <= Act.get((c, week, time_slot), 0), f"Classroom_Availability_{c}_{s}"

    # A.17 - Consistency of class hour assignments
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g in class_courses[s]:
                    model += z[c, s] >= w[c, s, g], f"Consistency_1_{c}_{s}_{g}"
                    model += w[c, s, g] <= 1 - z_v[s], f"Consistency_1_zv_{c}_{s}_{g}"

    # A.18 - Consistency of class hour assignments
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                model += z[c, s] <= lpSum(w[c, s, g] for g in class_courses[s]), f"Consistency_2_{c}_{s}"

    # A.19 - Ensure different classes do not share classrooms
    for c in classrooms:
        for s in subjects:
            if Gs.get(s, 0) == 1:
                for g1 in class_courses[s]:
                    for g2 in class_courses[s]:
                        if g1 != g2:
                            model += w[c, s, g1] + w[c, s, g2] <= 1, f"No_Shared_Classroom_{c}_{s}_{g1}_{g2}"

    # A.20 - Computer room requirements
    for c in classrooms:
        for s in subjects:
            if Cs.get(s, 0) == 1 and c not in Pc:
                model += z[c, s] == 0, f"Computer_Requirement_{c}_{s}"

    # A.21 - Desk requirements
    for c in classrooms:
        for s in subjects:
            if Ts.get(s, 0) == 1 and Tc.get(c, 0) == 0:  # Course requires desks but classroom does not have them
                model += z[c, s] == 0, f"Table_Requirement_{c}_{s}"

    # A.22 - Isolated classrooms
    for c in classrooms:
        for s in subjects:
            if Ic.get(c, 0) == 1:
                model += lpSum(z[c_prime, s] for c_prime in classrooms if c_prime != c) <= (1 - z[c, s]) * len(classrooms), f"Isolation_Requirement_{c}_{s}"

    # 3. Solve
    import pulp
    solver = pulp.PULP_CBC_CMD(
        timeLimit=10,
        gapRel=0.05,
        threads=1,
        cuts='on',
    )
    model.solve(solver)

    # Output optimization results
    print("Week:", week)
    print("Time Slot:", time_slot)
    print("Optimization Status:", LpStatus[model.status])
    value = model.objective.value()
    print("Objective Function Value:", value)
    if model.status == 1:  # Output assignment details only if a feasible solution is found
        print("\nClassroom Assignment Details:")
        assigned_classrooms = []
        if z_v[s].value() == 1:
            assigned_classrooms.append((0, s))
            print(f" - Virtual Classroom assigned to course {s}")
        for (c, s), var in z.items():
            if var.value() == 1:
                assigned_classrooms.append((c, s))
                print(f" - Classroom {c} assigned to course {s}")
    else:
        print("\n!! No feasible solution found, please check constraints !!")
        # Output conflicting constraints (if any)
        for name, constraint in model.constraints.items():
            if not constraint.valid():
                print(f"Conflicting Constraint: {name}")

    # Collect and write results within the loop
    if model.status == 1:
        # Collect results for the current iteration
        value = model.objective.value()
        assignment_pairs = []
        for (c, s), var in z.items():
            if var.value() == 1:
                assignment_pairs.append((s, c))
                print(f" - Classroom {c} assigned to course {s}")
        for s in subjects:
            if z_v[s].value() == 1:
                assignment_pairs.append((s, 0))  # Virtual classroom
        
        # Generate the result string
        sorted_assignments = sorted(assignment_pairs, key=lambda x: (int(x[0]), int(x[1])))
        result_entries = [f"{{{s}:{c}}}" for s, c in sorted_assignments]
        result_str = ",".join(result_entries)

        # Generate subject_candidates (original logic retained)
        subject_set = {s for s, _ in sorted_assignments}
        sorted_subjects = sorted(subject_set, key=int)
        subject_candidates = "{%s}" % ",".join(map(str, sorted_subjects))
    else:
        subject_candidates = "{}"
        result_str = "{}"

    # Write to CSV (retain original writing logic)
    with open("results-two_changes_combined.csv", "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow([
            df.iloc[count]["Week"],
            df.iloc[count]["Time_slot"],
            subject_candidates,
            result_str,
            value
        ])
        real_usage = sum(z[c, s].varValue for c in classrooms for s in subjects)
    virtual_usage = sum(z_v[s].varValue for s in subjects)
    print(f"Actual classroom usage: {real_usage}, Virtual classroom usage: {virtual_usage}, Total cost: {real_usage + beta * virtual_usage}")

    results.append({
        "Week": week,
        "Time_slot": time_slot,
        "subject_candidates": subject_candidates,
        "result": result_str,
        "value": value,
        "real_usage": real_usage,
        "virtual_usage": virtual_usage,
        "total_cost": real_usage + beta * virtual_usage
    })


Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/anaconda3/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/arm64/cbc /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/ab345e9c70764517ada68fed929a76be-pulp.mps -sec 10 -gomory on knapsack on probing on -ratio 0.05 -threads 1 -timeMode elapsed -branch -printingOptions all -solution /var/folders/xg/6886p3rj5kb44k2smttjctf80000gn/T/ab345e9c70764517ada68fed929a76be-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 1141 COLUMNS
At line 4733 RHS
At line 5870 BOUNDS
At line 6215 ENDATA
Problem MODEL has 1136 rows, 346 columns and 2785 elements
Coin0008I MODEL read with 0 errors
seconds was changed from 1e+100 to 10
Option for gomoryCuts changed from ifmove to on
Option for knapsackCuts changed from ifmove to on
ratioGap was changed from 0 to 0.05
threads was changed from 0 to 1
Option for timeMode changed from cpu to elapsed
Continuous objective value is 0.196463 

In [9]:
results_df = pd.DataFrame(results)
results_df


,Week,Time_slot,subject_candidates,result,value,real_usage,virtual_usage,total_cost
0,1,1,"{11,13}","{11:4},{11:58},{13:32}",3.0,3.0,0.0,3.0
1,1,2,"{5,16,18,40}","{5:11},{16:15},{16:27},{16:28},{16:33},{16:35}...",11.0,11.0,0.0,11.0
2,1,3,{35},{35:55},1.0,1.0,0.0,1.0
3,1,4,{53},"{53:8},{53:11},{53:49},{53:55}",4.0,4.0,0.0,4.0
4,1,5,{35},{35:15},1.0,1.0,0.0,1.0
...,...,...,...,...,...,...,...,...
89,12,1,{2},"{2:29},{2:58}",2.0,2.0,0.0,2.0
90,12,2,{33},{33:14},1.0,1.0,0.0,1.0
91,12,3,{27},{27:49},1.0,1.0,0.0,1.0
92,12,7,{30},"{30:11},{30:49}",2.0,2.0,0.0,2.0


In [10]:
# Count the total usage of virtual classrooms
print("Total virtual classroom usage:", sum(results_df["virtual_usage"]))
# Count the total usage of actual classrooms
print("Total actual classroom usage:", sum(results_df["real_usage"]))
# Count the total cost
print("Total cost:", sum(results_df["total_cost"]))

虚拟教室使用总量: 5.0
实际教室使用总量: 271.0
总成本: 321.0
